In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

text_english = open("tiny-shakespeare.txt").read()
text_turkish = open("tiny-shakespeare_tr.txt").read()

unique_chars_english = sorted(list(set(text_english)))
unique_chars_turkish = sorted(list(set(text_turkish)))

stoi_en = {ch:i for i, ch in enumerate(unique_chars_english)}
itos_en = {i:ch for i, ch in enumerate(unique_chars_english)}

stoi_tr = {ch:i for i, ch in enumerate(unique_chars_turkish)}
itos_tr = {i:ch for i, ch in enumerate(unique_chars_turkish)}

encode_en = lambda s: [stoi_en[ch] for ch in s]
decode_en = lambda ary: [itos_en[i] for i in ary]

encode_tr = lambda s: [stoi_tr[ch] for ch in s]
decode_tr = lambda ary: [itos_tr[i] for i in ary]

encoded_all_data_en = encode_en(text_english)
encoded_all_data_tensor_en = torch.tensor(encoded_all_data_en)
n_en = int(0.8 * len(encoded_all_data_tensor_en))
train_data_tensor_en = encoded_all_data_tensor_en[:n_en]
val_data_tensor_en = encoded_all_data_tensor_en[n_en:]

#send to GPU
train_data_tensor_en = train_data_tensor_en.to(device)
val_data_tensor_en = val_data_tensor_en.to(device)

# encoded_all_data_tr = encode_tr(text_turkish)
# encoded_all_data_tensor_tr = torch.tensor(encoded_all_data_tr)
# n_tr = int(0.8 * len(encoded_all_data_tensor_tr))
# train_data_tensor_tr = encoded_all_data_tensor_tr[:n_tr]
# val_data_tensor_tr = encoded_all_data_tensor_tr[n_tr:]
# train_data_tensor_tr = train_data_tensor_tr.to(device)
# val_data_tensor_tr = val_data_tensor_tr.to(device)

In [ ]:
class AttentionHead(nn.Module):

    def __init__(self, num_embedding_C, head_size_Hs, max_block_size_T, masked=True, dropout=0.2) -> None:
        super().__init__()
        self.num_embedding_C = num_embedding_C
        self.head_size_Hs = head_size_Hs
        self.masked = masked
        # Query can come from a different context
        self.w_query = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size_Hs, bias=False)
        # Key and Value should come from the same source
        self.w_key = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size_Hs, bias=False)
        self.w_value = nn.Linear(in_features=self.num_embedding_C, out_features=self.head_size_Hs, bias=False)
        self.tril: torch.Tensor
        self.register_buffer('tril', torch.tril(torch.ones(max_block_size_T, max_block_size_T)))
        self.dropout = nn.Dropout(dropout)


    def forward(self, source_x, target_memory_context=None, key_padding_mask=None):
        if target_memory_context is None:
            self.Q = self.w_query(source_x) # self attention [B, T_k, C] x [C, Hs] = [B, T_k, Hs]
        else:
            self.Q = self.w_query(target_memory_context) # cross-attention  # [B, T_q, C] x [C, Hs] = [B, T_q, Hs]

        self.K = self.w_key(source_x) # [B, T_k, C] x [C, Hs] = [B, T_k, Hs]
        self.V = self.w_value(source_x) # [B, T_k, Hs]

        dim_k = self.K.shape[-1] # = Hs
        inv_sqrt_dim_k = (dim_k ** -0.5)
        attention_scores = (self.Q @ self.K.transpose(-2, -1)) * inv_sqrt_dim_k # [B, T_q, Hs] @ [B, Hs, T_k] = [B, T_q, T_k]

        if self.masked: # self-attention
            T_q, T_k = attention_scores.shape[-2:]
            attention_scores = attention_scores.masked_fill(self.tril[:T_q, :T_k] == 0, float('-inf')) # masked self-attention

        attention_map = F.softmax(attention_scores, dim=-1) #[B, T_q, T_k]
        attention_map = self.dropout(attention_map)

        z = attention_map @ self.V # [B, T_q, T_k] @ [B, T_k, Hs] = [B, T_q, Hs]

        return z

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, num_embedding_C, head_size_Hs, max_block_size_T, masked, dropout=0.2) -> None:
        super().__init__()
        self.num_heads = num_heads
        self.attention_heads = nn.ModuleList([AttentionHead(num_embedding_C, head_size_Hs, max_block_size_T, masked, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(in_features=num_heads*head_size_Hs, out_features=num_embedding_C)
        self.dropout = nn.Dropout(dropout)

    def forward(self, source_x, target_memory_context=None, key_padding_mask=None):
        x = torch.cat([h(source_x, target_memory_context, key_padding_mask) for h in self.attention_heads], dim=-1) #cat([B, T, Hs], [B, T, Hs], [B, T, Hs] ...num_heads) cat on Hs, the last column = -1
        out = self.proj(x) # B, T, Hs*num_heads] @ [Hs*num_heads, C] = [B, T, C]
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):

    def __init__(self, num_embedding_C, dropout=0.2) -> None:
        super().__init__()
        self.ff = nn.Sequential(nn.Linear(num_embedding_C, num_embedding_C),
                                nn.ReLU(),
                                nn.Linear(num_embedding_C, num_embedding_C),
                                nn.Dropout(dropout))

    def forward(self, X):
        #Input X from MultiHeadAttention output [B, T, C]
        return self.ff(X)

class DecoderResidualBlock(nn.Module): #the masked is True for decoder, no information from the future sequences

    def __init__(self, num_heads, num_embedding_C, head_size_Hs, max_block_size_T, dropout=0.2) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(num_heads, num_embedding_C, head_size_Hs, max_block_size_T, True, dropout)
        self.ffwd = FeedForward(num_embedding_C, dropout)
        self.layer_norm1 = nn.LayerNorm(num_embedding_C)
        self.layer_norm2 = nn.LayerNorm(num_embedding_C)

    def forward(self, X):
        #Input is Token_Embeddings + Positional Embeddings [B, T, C]
        x =  X + self.mha(self.layer_norm1(X))
        x =  x + self.ffwd(self.layer_norm2(x))
        return x


class Translator(nn.Module):
    def __init__(self, vocab_size_en, vocab_size_tr, batch_size_B, block_size_T, num_embedding_C, attention_head_size_Hs, num_multi_attention_head_Hn, num_transformer_layer_Nx, dropout = 0.2, learning_rate=3e-4):
        super().__init__()
        self.vocab_size_en = vocab_size_en
        self.batch_size_B = batch_size_B
        self.block_size_T = block_size_T
        self.num_embedding_C = num_embedding_C
        self.attention_head_size_Hs = attention_head_size_Hs
        self.num_multi_attention_head_Hn = num_multi_attention_head_Hn
        self.num_transformer_layer_Nx = num_transformer_layer_Nx
        self.dropout = dropout
        self.learning_rate = learning_rate

        self.token_embeddings_en = nn.Embedding(num_embeddings=self.vocab_size_en, embedding_dim=self.num_embedding_C)
        self.position_embeddings_en = nn.Embedding(num_embeddings=self.block_size_T, embedding_dim=self.num_embedding_C)
        self.decoder_blocks = nn.Sequential(
        self.num_heads = num_multi_attention_head_Hn
        self.head_size_Hs = attention_head_size_Hs
        self.max_block_size_T = block_size_T
        self.decoder_blocks = nn.Sequential(
            *[DecoderResidualBlock(self.num_heads, self.num_embedding_C, self.head_size_Hs, self.max_block_size_T, self.dropout) for _ in range(self.num_transformer_layer_Nx)]
            )
    def forward(self, X_idx_en, Y_targets=None):
        batch_size, sequence_length = X_idx_en.shape
        output_token_embedding = self.token_embeddings_en(X_idx_en) # [B, T, C]
        positions = torch.arange(sequence_length, device=X_idx_en.device)
        output_position_embedding = self.position_embeddings_en(positions) # [T, C]

        x = output_token_embedding + output_position_embedding
        x = self.decoder_blocks(x)
        x = output_token_embedding + output_position_embedding
        x = self.decoder_blocks(x)
        # x = nn.Blocks(num_layer, head_size)
            # x = multiHeadAttention(x, num_layer, head_size)
                #SelfAttention
            # Add
        logits = self.last_linear_head(x) # [B, T, V = vocab_size]
        logits = logits.reshape(batch_size * sequence_length, self.vocab_size_en)

        loss = None
        if Y_targets is not None:
            Y_targets = Y_targets.reshape(batch_size * sequence_length)
            loss = F.cross_entropy(logits, Y_targets)

        return logits, loss
        #send to loss function
        loss = F.cross_entropy(logits, Y_targets) #[B*T, V] vs [B*T]

        return logits, loss








In [19]:
B = 4
T = 2
Hs = 5
a = torch.ones(B, T, Hs)
b = torch.ones(B, T, Hs)*2
c = torch.ones(B, T, Hs)*3
print(a)
print(b)
print(c)

modules = [a, b, c]

cats = torch.cat(modules, dim=-1)
print(cats.shape)




tensor([[[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]],

        [[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]],

        [[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]],

        [[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]])
tensor([[[2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.]],

        [[2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.]],

        [[2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.]],

        [[2., 2., 2., 2., 2.],
         [2., 2., 2., 2., 2.]]])
tensor([[[3., 3., 3., 3., 3.],
         [3., 3., 3., 3., 3.]],

        [[3., 3., 3., 3., 3.],
         [3., 3., 3., 3., 3.]],

        [[3., 3., 3., 3., 3.],
         [3., 3., 3., 3., 3.]],

        [[3., 3., 3., 3., 3.],
         [3., 3., 3., 3., 3.]]])
torch.Size([4, 2, 15])
